In [33]:

import pufferlib
import pufferlib.vector
import torch
import torch.nn as nn
import numpy as np
device = device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
from stable_baselines3.common.buffers import ReplayBuffer
import pufferlib.vector
from pufferlib.pufferl import load_config, load_env
from pufferlib.ocean.breakout.breakout import Breakout
import time
DEBUG_MODE = True
DEBUG_FREQ = 1000

def debug_log(step, obs, actions, rewards, q_values=None, loss=None):
    print(f"\n[DEBUG] Step {step}")

    if obs is not None:
        print("Obs stats:",
              f"min={obs.min():.4f}",
              f"max={obs.max():.4f}",
              f"mean={obs.mean():.4f}")

    if actions is not None:
        unique, counts = np.unique(actions, return_counts=True)
        print("Action distribution:", dict(zip(unique, counts)))

    if rewards is not None:
        print("Reward stats:",
              f"min={rewards.min():.4f}",
              f"max={rewards.max():.4f}",
              f"mean={rewards.mean():.4f}")

    if q_values is not None:
        print("Q-values:",
              f"mean={q_values.mean().item():.4f}",
              f"std={q_values.std().item():.4f}")

    if loss is not None:
        print("Loss:", f"{loss:.6f}")

Using device: cuda


In [34]:



def make_env(num_envs=5):
    vecenv = pufferlib.vector.make(
        Breakout,
        num_envs=num_envs,
        backend=pufferlib.vector.Multiprocessing
    )
    return vecenv


def make_buffer(buffer_size, vecenv):
    replay_buffer = ReplayBuffer(
        n_envs=vecenv.num_environments,
        buffer_size=buffer_size,
        observation_space=vecenv.single_observation_space,
        action_space=vecenv.single_action_space,
        handle_timeout_termination=False,
        device='cpu'
    )
    return replay_buffer


class QNetwork(nn.Module):
    def __init__(self, input_dim=118, num_actions=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            #nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(512, 512),
            #nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(512, 256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, num_actions)
        )

    def forward(self, x):
        return self.net(x)


def epsilon_greedy_policy(obs, q_network, n_actions, num_envs, epsilon, device):
    obs_tensor = torch.from_numpy(obs).float().to(device)
    with torch.no_grad():
        q_values = q_network(obs_tensor)
        greedy_actions = torch.argmax(q_values, dim=1).cpu().numpy()

    random_actions = np.random.randint(0, n_actions, size=num_envs)
    take_random = np.random.rand(num_envs) < epsilon

    actions = np.where(take_random, random_actions, greedy_actions)
    return actions


def train_on_mini_batch(replay_buffer, q_network, q_target, batch_size, optimizer, loss_criterion, gamma, device):
    # print("Using GPU:", next(q_network.parameters()).is_cuda)
    # print("CUDA mem:", torch.cuda.memory_allocated(device))
    mini_batch = replay_buffer.sample(batch_size=batch_size)
    obs = mini_batch.observations.to(device)
    next_obs = mini_batch.next_observations.to(device)
    actions = mini_batch.actions.to(device).squeeze(-1)
    rewards = mini_batch.rewards.to(device).squeeze(-1)
    dones = mini_batch.dones.to(device).squeeze(-1).float()  # Ensure float for math ops

    q_values = q_network(obs)
    q_values_actions = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        target_q = q_target(next_obs)
        max_target_q = torch.max(target_q, dim=1)[0]
        target = rewards + gamma * max_target_q * (1 - dones)

    loss = loss_criterion(q_values_actions, target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()


def train(env, replay_buffer, total_steps, warmup_steps, target_update_freq, train_freq,
          q_network, q_target, batch_size, initial_epsilon, final_epsilon, epsilon_decay_steps, lr, gamma, device):

    optimizer = torch.optim.Adam(q_network.parameters(), lr=lr)
    loss_criterion = torch.nn.MSELoss()

    obs, _ = env.reset()

    for step in range(total_steps):
        epsilon = final_epsilon + ((initial_epsilon - final_epsilon) * (1 - (step / epsilon_decay_steps)))
        epsilon = max(epsilon, final_epsilon)

        if step % 1000 == 0:
            print(f"\nStep: {step}, Epsilon: {epsilon:.4f}")

        actions = epsilon_greedy_policy(obs, q_network, env.single_action_space.n, env.num_envs, epsilon, device)
        next_obs, rewards, dones, truncs, infos = env.step(actions)

        n, avg_episode_return, avg_episode_length = 0, 0, 0
        for item in infos:
            if "episode_length" in item:
                n += 1
                avg_episode_length += item['episode_length']
                avg_episode_return += item['episode_return']
        #if n != 0:
            #print(f"Avg Ep Length: {avg_episode_length / n:.2f}, Avg Return: {avg_episode_return / n:.2f}")

        replay_buffer.add(obs, next_obs, actions, rewards, dones, infos=None)

        loss = None
        q_val_snapshot = None
        if step >= warmup_steps and step % train_freq == 0:
            mini_batch = replay_buffer.sample(batch_size=batch_size)
            obs_tensor = mini_batch.observations.to(device)
            q_val_snapshot = q_network(obs_tensor).detach()

            loss = train_on_mini_batch(
                replay_buffer=replay_buffer,
                q_network=q_network,
                q_target=q_target,
                batch_size=batch_size,
                optimizer=optimizer,
                loss_criterion=loss_criterion,
                gamma=gamma,
                device=device
            )

        if step % target_update_freq == 0:
            q_target.load_state_dict(q_network.state_dict())

        if DEBUG_MODE and step % DEBUG_FREQ == 0:
            debug_log(step, obs, actions, rewards, q_values=q_val_snapshot, loss=loss)

        obs = next_obs



In [ ]:
breakoutEnv = make_env(num_envs=10)

replay_buffer = make_buffer(buffer_size=500_000, vecenv=breakoutEnv)

q_network = QNetwork(input_dim=118, num_actions=4).to(device)
q_target = QNetwork(input_dim=118, num_actions=4).to(device)
q_target.load_state_dict(q_network.state_dict())

total_steps = 500_000                # More steps for Atari games
epsilon_decay_steps = 250_000        # Slower decay over time
target_update_freq = 5_000           # More stable targets
train_freq = 4                       # More frequent training
lr = 5e-5                            # Lower LR for stability
initial_epsilon = 1.0
final_epsilon = 0.05
warmup_steps = 10_000                # Let the buffer fill up a bit
batch_size = 64                     # Larger batch size helps with noisy envs
gamma = 0.99                         # Standard for Atari

Process Process-40:
Process Process-34:
Process Process-37:
Process Process-38:
Process Process-39:
Process Process-31:
Process Process-33:
Process Process-32:
Process Process-35:
Process Process-36:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/venv/main/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/venv/main/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/venv/main/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/venv/main/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/venv/main/lib/python3.12/multipro

In [39]:
obs, _ = breakoutEnv.reset()
for _ in range(100):
    actions = breakoutEnv.action_space.sample()
    obs, reward, done, trunc, info = breakoutEnv.step(actions)
    if np.sum(reward) > 0:
        print(reward)

[0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 1. 1. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]


In [40]:
train(env = breakoutEnv,replay_buffer = replay_buffer, total_steps = total_steps, warmup_steps = warmup_steps,
      target_update_freq=target_update_freq, train_freq=train_freq,q_network=q_network, q_target = q_target, 
      batch_size = batch_size,initial_epsilon=initial_epsilon, final_epsilon=final_epsilon, 
      epsilon_decay_steps=epsilon_decay_steps, lr = lr, gamma = gamma, device = device)


Step: 0, Epsilon: 1.0000

[DEBUG] Step 0
Obs stats: min=-0.0069 max=1.0000 mean=0.0379
Action distribution: {0: 2, 1: 7, 2: 1}
Reward stats: min=0.0000 max=0.0000 mean=0.0000



Step: 1000, Epsilon: 0.9962

[DEBUG] Step 1000
Obs stats: min=-0.0069 max=1.0000 mean=0.0405
Action distribution: {0: 2, 1: 3, 2: 5}
Reward stats: min=0.0000 max=0.0000 mean=0.0000

Step: 2000, Epsilon: 0.9924

[DEBUG] Step 2000
Obs stats: min=-0.0073 max=1.0000 mean=0.0383
Action distribution: {1: 6, 2: 4}
Reward stats: min=0.0000 max=0.0000 mean=0.0000

Step: 3000, Epsilon: 0.9886

[DEBUG] Step 3000
Obs stats: min=-0.0074 max=1.0000 mean=0.0415
Action distribution: {0: 3, 1: 4, 2: 2, 3: 1}
Reward stats: min=0.0000 max=0.0000 mean=0.0000

Step: 4000, Epsilon: 0.9848

[DEBUG] Step 4000
Obs stats: min=-0.0079 max=1.0000 mean=0.0416
Action distribution: {1: 1, 2: 8, 3: 1}
Reward stats: min=0.0000 max=1.0000 mean=0.1000

Step: 5000, Epsilon: 0.9810

[DEBUG] Step 5000
Obs stats: min=-0.0080 max=1.0000 mean=0.0371
Action distribution: {0: 1, 1: 4, 2: 5}
Reward stats: min=0.0000 max=0.0000 mean=0.0000


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 train(env = breakoutEnv,replay_buffer = replay_buffer, total_steps = total_steps, warmup     │
│   2 │     target_update_freq=target_update_freq, train_freq=train_freq,q_network=q_network,      │
│   3 │     batch_size = batch_size,initial_epsilon=initial_epsilon, final_epsilon=final_epsil     │
│   4 │     epsilon_decay_steps=epsilon_decay_steps, lr = lr, gamma = gamma, device = device)      │
│                                                                                                  │
│ in train:104                                                                                     │
│                                                                                                  │
│   101 │   │   │   print(f"\nStep: {step}, Epsilon: {epsilon:.4f}")                               │
│   102 │   │                                                                                      │
│   103 │   │   actions = epsilon_greedy_policy(obs, q_network, env.single_action_space.n, env.n   │
│ ❱ 104 │   │   next_obs, rewards, dones, truncs, infos = env.step(actions)                        │
│   105 │   │                                                                                      │
│   106 │   │   n, avg_episode_return, avg_episode_length = 0, 0, 0                                │
│   107 │   │   for item in infos:                                                                 │
│                                                                                                  │
│ /root/backtoRL/.venv/lib/python3.12/site-packages/pufferlib/vector.py:49 in step                 │
│                                                                                                  │
│    46 def step(vecenv, actions):                                                                 │
│    47 │   actions = np.asarray(actions)                                                          │
│    48 │   vecenv.send(actions)                                                                   │
│ ❱  49 │   obs, rewards, terminals, truncations, infos, env_ids, masks = vecenv.recv()            │
│    50 │   return obs, rewards, terminals, truncations, infos # include env_ids or no?            │
│    51                                                                                            │
│    52 class Serial:                                                                              │
│                                                                                                  │
│ /root/backtoRL/.venv/lib/python3.12/site-packages/pufferlib/vector.py:375 in recv                │
│                                                                                                  │
│   372 │   │   │   │   │   self.waiting_workers.append(worker)                                    │
│   373 │   │   │                                                                                  │
│   374 │   │   │   if sem == INFO:                                                                │
│ ❱ 375 │   │   │   │   self.infos[worker] = self.recv_pipes[worker].recv()                        │
│   376 │   │   │                                                                                  │
│   377 │   │   │   if not self.ready_workers:                                                     │
│   378 │   │   │   │   continue                                                                   │
│                                                                                                  │
│ /venv/main/lib/python3.12/multiprocessing/connection.py:250 in recv                              │
│                                                            